In [16]:
# CELL 1
import os

os.environ["GEMINI_API_KEY"] = "apikey"

In [17]:
# CELL 2
import importlib
import utils

importlib.reload(utils)

<module 'utils' from 'D:\\chatgpt api\\utils.py'>

In [18]:
import os
from google import genai


def get_completion_from_messages(
    messages,
    model="gemini-3.6-flash",
    temperature=0.7,
    max_tokens=1000
):
    # ---------------------------------------------------------
    # GEMINI API KEY
    # ---------------------------------------------------------
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

    if not GEMINI_API_KEY:
        raise RuntimeError(
            "GEMINI_API_KEY is not set. "
            "Please set your Gemini API key before running."
        )

    # Create Gemini client
    client = genai.Client(api_key=GEMINI_API_KEY)

    # ---------------------------------------------------------
    # PROCESS MESSAGES
    # ---------------------------------------------------------
    formatted_messages = []
    system_message = ""

    for message in messages:

        if "content" not in message:
            raise ValueError(
                f"Message missing 'content' field: {message}"
            )

        role = message["role"]
        content = message["content"]

        # Save system instruction separately
        if role == "system":
            system_message = content

        elif role == "user":
            formatted_messages.append({
                "role": "user",
                "parts": [
                    {
                        "text": content
                    }
                ]
            })

        elif role == "assistant":
            formatted_messages.append({
                "role": "model",
                "parts": [
                    {
                        "text": content
                    }
                ]
            })

        else:
            raise ValueError(
                f"Unsupported message role: {role}"
            )

    # ---------------------------------------------------------
    # ADD SYSTEM MESSAGE TO FIRST USER MESSAGE
    # ---------------------------------------------------------
    if formatted_messages and system_message:

        # Find first user message
        for message in formatted_messages:
            if message["role"] == "user":

                original_text = message["parts"][0]["text"]

                message["parts"][0]["text"] = (
                    system_message
                    + "\n\n"
                    + original_text
                )

                break

    # ---------------------------------------------------------
    # GENERATE RESPONSE
    # ---------------------------------------------------------
    response = client.models.generate_content(
        model=model,
        contents=formatted_messages,
        config={
            "temperature": temperature,
            "max_output_tokens": max_tokens
        }
    )

    return response.text

#### Get the relevant products and categories
Here is the list of products and categories that are in the product catalog.

In [19]:
products_and_category = utils.get_products_and_category()
products_and_category

Enter your product/category question:  tv


[{'category': 'Televisions and Home Theater Systems',
  'products': ['CineView 4K TV',
   'CineView 8K TV',
   'CineView OLED TV',
   'SoundMax Home Theater',
   'SoundMax Soundbar']}]

In [20]:
import importlib
import utils

importlib.reload(utils)

print(hasattr(utils, "get_products_and_category"))

True


### Find relevant product and category names (version 1)
This could be the version that is running in production.

In [21]:
def find_category_and_product_v1(user_input,products_and_category):

    delimiter = "####"
    system_message = f"""
    You will be provided with customer service queries. \
    The customer service query will be delimited with {delimiter} characters.
    Output a python list of json objects, where each object has the following format:
        'category': <one of Computers and Laptops, Smartphones and Accessories, Televisions and Home Theater Systems, \
    Gaming Consoles and Accessories, Audio Equipment, Cameras and Camcorders>,
    AND
        'products': <a list of products that must be found in the allowed products below>


    Where the categories and products must be found in the customer service query.
    If a product is mentioned, it must be associated with the correct category in the allowed products list below.
    If no products or categories are found, output an empty list.
    

    List out all products that are relevant to the customer service query based on how closely it relates
    to the product name and product category.
    Do not assume, from the name of the product, any features or attributes such as relative quality or price.

    The allowed products are provided in JSON format.
    The keys of each item represent the category.
    The values of each item is a list of products that are within that category.
    Allowed products: {products_and_category}
    

    """
    
    few_shot_user_1 = """I want the most expensive computer."""
    few_shot_assistant_1 = """ 
    [{'category': 'Computers and Laptops', \
'products': ['TechPro Ultrabook', 'BlueWave Gaming Laptop', 'PowerLite Convertible', 'TechPro Desktop', 'BlueWave Chromebook']}]
    """
    
    messages =  [  
    {'role':'system', 'content': system_message},    
    {'role':'user', 'content': f"{delimiter}{few_shot_user_1}{delimiter}"},  
    {'role':'assistant', 'content': few_shot_assistant_1 },
    {'role':'user', 'content': f"{delimiter}{user_input}{delimiter}"},  
    ] 
    return get_completion_from_messages(messages)


### Evaluate on some queries

In [22]:
customer_msg_0 = f"""Which TV can I buy if I'm on a budget?"""

products_by_category_0 = find_category_and_product_v1(customer_msg_0,
                                                      products_and_category)
print(products_by_category_0)

[
    {
        'category': 'Televisions and Home Theater Systems',
        'products': [
            'CineView 4K TV',
            'CineView 8K TV',
            'CineView OLED TV'
        ]
    }
]


In [23]:
customer_msg_1 = f"""I need a charger for my smartphone"""

products_by_category_1 = find_category_and_product_v1(customer_msg_1,
                                                      products_and_category)
print(products_by_category_1)

```json
[
    {
        "category": "Smartphones and Accessories",
        "products": [
            "ChargX Wireless Charger",
            "WallCharge Adapter",



In [24]:
customer_msg_2 = f"""
What computers do you have?"""

products_by_category_2 = find_category_and_product_v1(customer_msg_2,
                                                      products_and_category)
products_by_category_2

"[\n    {\n        'category': 'Computers and Laptops',\n        'products': [\n            'TechPro Ultrabook',\n            'BlueWave Gaming Laptop',\n            'PowerLite Convertible',\n            'TechPro Desktop',\n            'BlueWave Chromebook'\n        ]\n    }\n]"

In [25]:
customer_msg_3 = f"""
tell me about the smartx pro phone and the fotosnap camera, the dslr one.
Also, what TVs do you have?"""

products_by_category_3 = find_category_and_product_v1(customer_msg_3,
                                                      products_and_category)
print(products_by_category_3)

[
    {
        "category": "Smartphones and Accessories",
        "products": [
            "SmartX Pro Phone"
        ]
    },
    {
        "category": "Cameras and Camcorders",
        "products": [
            "FotoSnap DSLR Camera"
        ]
    },
    {
        "category": "Televisions and Home Theater Systems",
        "products": [
            "CineView 4K TV",
            "CineView 8K TV",
            "CineView OLED TV"
        ]
    }
]


### Harder test cases
Identify queries found in production, where the model is not working as expected.

In [26]:
customer_msg_4 = f"""
tell me about the CineView TV, the 8K one, Gamesphere console, the X one.
I'm on a budget, what computers do you have?"""

products_by_category_4 = find_category_and_product_v1(customer_msg_4,
                                                      products_and_category)
print(products_by_category_4)

```json
[
    {
        "category": "Televisions and Home Theater Systems",
        "products": [
            "CineView 8K


### Modify the prompt to work on the hard test cases

In [27]:
def find_category_and_product_v2(user_input,products_and_category):
    """
    Added: Do not output any additional text that is not in JSON format.
    Added a second example (for few-shot prompting) where user asks for 
    the cheapest computer. In both few-shot examples, the shown response 
    is the full list of products in JSON only.
    """
    delimiter = "####"
    system_message = f"""
    You will be provided with customer service queries. \
    The customer service query will be delimited with {delimiter} characters.
    Output a python list of json objects, where each object has the following format:
        'category': <one of Computers and Laptops, Smartphones and Accessories, Televisions and Home Theater Systems, \
    Gaming Consoles and Accessories, Audio Equipment, Cameras and Camcorders>,
    AND
        'products': <a list of products that must be found in the allowed products below>
    Do not output any additional text that is not in JSON format.
    Do not write any explanatory text after outputting the requested JSON.


    Where the categories and products must be found in the customer service query.
    If a product is mentioned, it must be associated with the correct category in the allowed products list below.
    If no products or categories are found, output an empty list.
    

    List out all products that are relevant to the customer service query based on how closely it relates
    to the product name and product category.
    Do not assume, from the name of the product, any features or attributes such as relative quality or price.

    The allowed products are provided in JSON format.
    The keys of each item represent the category.
    The values of each item is a list of products that are within that category.
    Allowed products: {products_and_category}
    

    """
    
    few_shot_user_1 = """I want the most expensive computer. What do you recommend?"""
    few_shot_assistant_1 = """ 
    [{'category': 'Computers and Laptops', \
'products': ['TechPro Ultrabook', 'BlueWave Gaming Laptop', 'PowerLite Convertible', 'TechPro Desktop', 'BlueWave Chromebook']}]
    """
    
    few_shot_user_2 = """I want the most cheapest computer. What do you recommend?"""
    few_shot_assistant_2 = """ 
    [{'category': 'Computers and Laptops', \
'products': ['TechPro Ultrabook', 'BlueWave Gaming Laptop', 'PowerLite Convertible', 'TechPro Desktop', 'BlueWave Chromebook']}]
    """
    
    messages =  [  
    {'role':'system', 'content': system_message},    
    {'role':'user', 'content': f"{delimiter}{few_shot_user_1}{delimiter}"},  
    {'role':'assistant', 'content': few_shot_assistant_1 },
    {'role':'user', 'content': f"{delimiter}{few_shot_user_2}{delimiter}"},  
    {'role':'assistant', 'content': few_shot_assistant_2 },
    {'role':'user', 'content': f"{delimiter}{user_input}{delimiter}"},  
    ] 
    return get_completion_from_messages(messages)


### Evaluate the modified prompt on the hard tests cases

In [28]:
customer_msg_3 = f"""
tell me about the smartx pro phone and the fotosnap camera, the dslr one.
Also, what TVs do you have?"""

products_by_category_3 = find_category_and_product_v2(customer_msg_3,
                                                      products_and_category)
print(products_by_category_3)

[
    {
        "category": "Smartphones and Accessories",
        "products": [
            "SmartX Pro Phone"
        ]
    },
    {


### Regression testing: verify that the model still works on previous test cases
Check that modifying the model to fix the hard test cases does not negatively affect its performance on previous test cases.

In [30]:
customer_msg_0 = f"""Which TV can I buy if I'm on a budget?"""

products_by_category_0 = find_category_and_product_v2(customer_msg_0,
                                                      products_and_category)
print(products_by_category_0)

[{"category": "Televisions and Home Theater Systems", "products": ["CineView 4K TV", "CineView 8K TV", "CineView OLED TV"]}]


### Gather development set for automated testing

In [31]:
msg_ideal_pairs_set = [
    
    # eg 0
    {'customer_msg':"""Which TV can I buy if I'm on a budget?""",
     'ideal_answer':{
        'Televisions and Home Theater Systems':set(
            ['CineView 4K TV', 'SoundMax Home Theater', 'CineView 8K TV', 'SoundMax Soundbar', 'CineView OLED TV']
        )}
    },

    # eg 1
    {'customer_msg':"""I need a charger for my smartphone""",
     'ideal_answer':{
        'Smartphones and Accessories':set(
            ['MobiTech PowerCase', 'MobiTech Wireless Charger', 'SmartX EarBuds']
        )}
    },
    # eg 2
    {'customer_msg':f"""What computers do you have?""",
     'ideal_answer':{
           'Computers and Laptops':set(
               ['TechPro Ultrabook', 'BlueWave Gaming Laptop', 'PowerLite Convertible', 'TechPro Desktop', 'BlueWave Chromebook'
               ])
                }
    },

    # eg 3
    {'customer_msg':f"""tell me about the smartx pro phone and \
    the fotosnap camera, the dslr one.\
    Also, what TVs do you have?""",
     'ideal_answer':{
        'Smartphones and Accessories':set(
            ['SmartX ProPhone']),
        'Cameras and Camcorders':set(
            ['FotoSnap DSLR Camera']),
        'Televisions and Home Theater Systems':set(
            ['CineView 4K TV', 'SoundMax Home Theater','CineView 8K TV', 'SoundMax Soundbar', 'CineView OLED TV'])
        }
    }, 
    
    # eg 4
    {'customer_msg':"""tell me about the CineView TV, the 8K one, Gamesphere console, the X one.
I'm on a budget, what computers do you have?""",
     'ideal_answer':{
        'Televisions and Home Theater Systems':set(
            ['CineView 8K TV']),
        'Gaming Consoles and Accessories':set(
            ['GameSphere X']),
        'Computers and Laptops':set(
            ['TechPro Ultrabook', 'BlueWave Gaming Laptop', 'PowerLite Convertible', 'TechPro Desktop', 'BlueWave Chromebook'])
        }
    },
    
    # eg 5
    {'customer_msg':f"""What smartphones do you have?""",
     'ideal_answer':{
           'Smartphones and Accessories':set(
               ['SmartX ProPhone', 'MobiTech PowerCase', 'SmartX MiniPhone', 'MobiTech Wireless Charger', 'SmartX EarBuds'
               ])
                    }
    },
    # eg 6
    {'customer_msg':f"""I'm on a budget.  Can you recommend some smartphones to me?""",
     'ideal_answer':{
        'Smartphones and Accessories':set(
            ['SmartX EarBuds', 'SmartX MiniPhone', 'MobiTech PowerCase', 'SmartX ProPhone', 'MobiTech Wireless Charger']
        )}
    },

    # eg 7 # this will output a subset of the ideal answer
    {'customer_msg':f"""What Gaming consoles would be good for my friend who is into racing games?""",
     'ideal_answer':{
        'Gaming Consoles and Accessories':set([
            'GameSphere X',
            'ProGamer Controller',
            'GameSphere Y',
            'ProGamer Racing Wheel',
            'GameSphere VR Headset'
     ])}
    },
    # eg 8
    {'customer_msg':f"""What could be a good present for my videographer friend?""",
     'ideal_answer': {
        'Cameras and Camcorders':set([
        'FotoSnap DSLR Camera', 'ActionCam 4K', 'FotoSnap Mirrorless Camera', 'ZoomMaster Camcorder', 'FotoSnap Instant Camera'
        ])}
    },
    
    # eg 9
    {'customer_msg':f"""I would like a hot tub time machine.""",
     'ideal_answer': []
    }
    
]


### Evaluate test cases by comparing to the ideal answers

In [32]:
import json
import re


def eval_response_with_ideal(response, ideal, debug=False):

    # ========================================================
    # STEP 1: CLEAN RESPONSE
    # ========================================================

    if response is None:
        response = ""

    response = str(response).strip()

    # Remove ```json and ``` if Gemini added them
    response = re.sub(
        r"```json",
        "",
        response,
        flags=re.IGNORECASE
    )

    response = response.replace(
        "```",
        ""
    ).strip()

    # ========================================================
    # STEP 2: EMPTY RESPONSE
    # ========================================================

    if not response:

        if ideal == []:
            return True

        if debug:
            print("Response is empty.")
            print("Ideal:", ideal)

        return False

    # ========================================================
    # STEP 3: TRY NORMAL JSON FIRST
    # ========================================================

    try:

        l_of_d = json.loads(response)

    except json.JSONDecodeError:

        l_of_d = None

    # ========================================================
    # STEP 4: TRY PYTHON-LIST STYLE
    # ========================================================

    if l_of_d is None:

        try:

            import ast

            l_of_d = ast.literal_eval(
                response
            )

        except Exception:

            l_of_d = None

    # ========================================================
    # STEP 5: EXTRACT JSON/PYTHON LIST FROM EXTRA TEXT
    # ========================================================

    if l_of_d is None:

        start = response.find("[")
        end = response.rfind("]")

        if (
            start != -1
            and end != -1
            and end > start
        ):

            extracted = response[
                start:end + 1
            ]

            # First try JSON
            try:

                l_of_d = json.loads(
                    extracted
                )

            except json.JSONDecodeError:

                # Then try Python syntax
                try:

                    import ast

                    l_of_d = ast.literal_eval(
                        extracted
                    )

                except Exception:

                    l_of_d = None

    # ========================================================
    # STEP 6: FAILED TO PARSE
    # ========================================================

    if l_of_d is None:

        if debug:

            print(
                "\n========== PARSING ERROR =========="
            )

            print(
                "\nRaw response:"
            )

            print(response)

            print(
                "\n===================================="
            )

        return False

    # ========================================================
    # STEP 7: MAKE SURE RESULT IS A LIST
    # ========================================================

    if not isinstance(
        l_of_d,
        list
    ):

        if debug:

            print(
                "\nExpected a list."
            )

            print(
                "Received:",
                type(l_of_d)
            )

            print(
                "Value:",
                l_of_d
            )

        return False

    # ========================================================
    # STEP 8: EMPTY LIST
    # ========================================================

    if (
        l_of_d == []
        and ideal == []
    ):

        return True

    # ========================================================
    # STEP 9: COMPARE RESULT WITH IDEAL
    # ========================================================

    if l_of_d == ideal:

        return True

    # ========================================================
    # STEP 10: DEBUG DIFFERENCE
    # ========================================================

    if debug:

        print(
            "\n========== EVALUATION =========="
        )

        print(
            "\nActual response:"
        )

        print(
            l_of_d
        )

        print(
            "\nIdeal response:"
        )

        print(
            ideal
        )

        print(
            "\nResult: False"
        )

        print(
            "================================"
        )

    return False

In [33]:
print(f'Customer message: {msg_ideal_pairs_set[7]["customer_msg"]}')
print(f'Ideal answer: {msg_ideal_pairs_set[7]["ideal_answer"]}')


Customer message: What Gaming consoles would be good for my friend who is into racing games?
Ideal answer: {'Gaming Consoles and Accessories': {'GameSphere VR Headset', 'ProGamer Controller', 'ProGamer Racing Wheel', 'GameSphere Y', 'GameSphere X'}}


In [34]:
response = find_category_and_product_v2(msg_ideal_pairs_set[7]["customer_msg"],
                                         products_and_category)
print(f'Resonse: {response}')

eval_response_with_ideal(response,
                              msg_ideal_pairs_set[7]["ideal_answer"])

Resonse: [
    {
        "category": "Gaming Consoles and Accessories",
        "products": [
            "GameSphere X",
            "HandHeld Games",
            "Game


False

### Run evaluation on all test cases and calculate the fraction of cases that are correct

In [35]:
# Note, this will not work if any of the api calls time out
score_accum = 0
for i, pair in enumerate(msg_ideal_pairs_set):
    print(f"example {i}")
    
    customer_msg = pair['customer_msg']
    ideal = pair['ideal_answer']
    
    # print("Customer message",customer_msg)
    # print("ideal:",ideal)
    response = find_category_and_product_v2(customer_msg,
                                                      products_and_category)
    
    
    # print("products_by_category",products_by_category)
    score = eval_response_with_ideal(response,ideal,debug=False)
    print(f"{i}: {score}")
    score_accum += score
    

n_examples = len(msg_ideal_pairs_set)
fraction_correct = score_accum / n_examples
print(f"Fraction correct out of {n_examples}: {fraction_correct}")

example 0
0: False
example 1
1: False
example 2
2: False
example 3
3: False
example 4
4: False
example 5
5: False
example 6
6: False
example 7
7: False
example 8


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}